# Jour 3 — Hyperopt CatBoost GPU + MLflow

**Meme objectif que 11c (Optuna)** : maximiser le **recall** tout en limitant
les faux positifs, mais avec **Hyperopt** (TPE bayesien).

Comparaison Optuna vs Hyperopt dans MLflow a la fin.

## Chargement des donnees

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable")

def _resolve(csv_path):
    csv_path = Path(csv_path)
    if csv_path.exists():
        return csv_path
    parquet_path = csv_path.with_suffix(".parquet")
    if parquet_path.exists():
        return parquet_path
    raise FileNotFoundError(f"Introuvable: {csv_path} ou {parquet_path}")

def _read(path, **csv_kwargs):
    path = Path(path)
    if path.suffix == ".parquet":
        cols = csv_kwargs.get("usecols")
        df = pd.read_parquet(path, columns=cols)
        nrows = csv_kwargs.get("nrows")
        if nrows:
            df = df.head(nrows)
        return df
    return pd.read_csv(path, **csv_kwargs)

root = _find_root("out")
path = _resolve(root / "out" / "accidents_model_ready_kept_with_time_bucket.csv")
print(f"Dataset: {path}")

TARGET = "grave"
SEP = ";"

product15_v2 = [
    "dep", "lum", "atm", "catr", "agg", "int", "circ", "col",
    "vma_bucket", "catv_family_4", "manv_mode", "driver_age_bucket",
    "choc_mode", "driver_trajet_family", "time_bucket",
]
cat_cols = product15_v2[:]
MISSING_CAT = "__MISSING__"

df = _read(path, sep=SEP)
X = df[product15_v2].copy()
y = df[TARGET].astype(int).copy()

for c in cat_cols:
    X[c] = X[c].astype("string").fillna(MISSING_CAT).astype(str)

n_neg = (y == 0).sum()
n_pos = (y == 1).sum()
print(f"Dataset : {len(y)} lignes")
print(f"Classe 0 (non grave) : {n_neg} ({100*n_neg/len(y):.1f}%)")
print(f"Classe 1 (grave)     : {n_pos} ({100*n_pos/len(y):.1f}%)")
print(f"Ratio naturel (neg/pos) : {n_neg/n_pos:.2f}")

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain : {X_train.shape} | Valid : {X_valid.shape}")

Dataset: /home/maxime/simplonalternance/alternance-CICDprediction/out/accidents_model_ready_kept_with_time_bucket.csv
Dataset : 164526 lignes
Classe 0 (non grave) : 105172 (63.9%)
Classe 1 (grave)     : 59354 (36.1%)
Ratio naturel (neg/pos) : 1.77

Train : (131620, 15) | Valid : (32906, 15)


## Configuration MLflow

In [4]:
import mlflow
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "hyperopt-catboost-recall"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print(f"MLflow experiment: {MLFLOW_EXPERIMENT}")

2026/02/25 14:25:00 INFO mlflow.tracking.fluent: Experiment with name 'hyperopt-catboost-recall' does not exist. Creating a new experiment.


MLflow experiment: hyperopt-catboost-recall


## Fonctions utilitaires

In [5]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, f1_score, fbeta_score,
    precision_score, recall_score, accuracy_score,
)

def find_best_threshold_fbeta(y_true, proba, beta=2.0):
    """Trouve le seuil qui maximise F-beta (beta>1 favorise recall)."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, proba)
    precisions = precisions[:-1]
    recalls = recalls[:-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        fbeta = ((1 + beta**2) * precisions * recalls) / (beta**2 * precisions + recalls)
    fbeta = np.nan_to_num(fbeta)
    best_idx = np.argmax(fbeta)
    return float(thresholds[best_idx]), float(fbeta[best_idx])


def compute_metrics(y_true, proba, threshold):
    preds = (proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba),
        "recall": recall_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds),
        "f2": fbeta_score(y_true, preds, beta=2),
        "accuracy": accuracy_score(y_true, preds),
    }

print("Fonctions utilitaires chargees")

Fonctions utilitaires chargees


## Hyperopt — search space GPU-compatible

Meme search space que le notebook Optuna (11c) :
- `depth` : 4-8
- `learning_rate` : 0.01-0.15 (log)
- `l2_leaf_reg` : 0.01-50 (log)
- `random_strength` : 0-10
- `min_data_in_leaf` : 1-100
- `border_count` : 64-255
- `scale_pos_weight` : 1.0-3.0
- `bootstrap_type` : Bayesian (+ temperature) ou MVS (+ subsample)

**Difference Hyperopt** : `fmin` minimise → on retourne `-PR_AUC`

In [6]:
import gc
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from catboost import CatBoostClassifier

RANDOM_SEED = 42
N_TRIALS = 30
MAX_ITERS = 4000
EARLY_STOP = 200
BETA = 2.0
MIN_PRECISION = 0.30

# --- Search space Hyperopt (GPU-safe) ---
space = {
    "depth": hp.quniform("depth", 4, 8, 1),
    "learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.15)),
    "l2_leaf_reg": hp.loguniform("l2_leaf_reg", np.log(0.01), np.log(50.0)),
    "random_strength": hp.uniform("random_strength", 0.0, 10.0),
    "min_data_in_leaf": hp.quniform("min_data_in_leaf", 1, 100, 1),
    "border_count": hp.quniform("border_count", 64, 255, 1),
    "scale_pos_weight": hp.uniform("scale_pos_weight", 1.0, 3.0),
    "bootstrap_type": hp.choice("bootstrap_type", [
        {
            "type": "Bayesian",
            "bagging_temperature": hp.uniform("bagging_temperature", 0.0, 2.0),
        },
        {
            "type": "MVS",
            "subsample": hp.uniform("subsample", 0.6, 1.0),
        },
    ]),
}


def objective(params):
    # Cast int pour les params discrets (Hyperopt retourne des float)
    depth = int(params["depth"])
    min_data_in_leaf = int(params["min_data_in_leaf"])
    border_count = int(params["border_count"])

    # Extraire bootstrap_type (nested dict dans Hyperopt)
    bt = params["bootstrap_type"]
    bootstrap_type = bt["type"]

    cb_params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": MAX_ITERS,
        "random_seed": RANDOM_SEED,
        "verbose": 0,
        "task_type": "GPU",
        "od_type": "Iter",
        "od_wait": EARLY_STOP,
        "depth": depth,
        "learning_rate": params["learning_rate"],
        "l2_leaf_reg": params["l2_leaf_reg"],
        "random_strength": params["random_strength"],
        "min_data_in_leaf": min_data_in_leaf,
        "border_count": border_count,
        "scale_pos_weight": params["scale_pos_weight"],
        "bootstrap_type": bootstrap_type,
        "grow_policy": "SymmetricTree",
    }
    if bootstrap_type == "Bayesian":
        cb_params["bagging_temperature"] = bt["bagging_temperature"]
    else:
        cb_params["subsample"] = bt["subsample"]

    model = CatBoostClassifier(**cb_params)
    model.fit(
        X_train, y_train,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
    )

    proba = model.predict_proba(X_valid)[:, 1]
    best_thr, _ = find_best_threshold_fbeta(y_valid, proba, beta=BETA)
    metrics = compute_metrics(y_valid, proba, best_thr)

    best_iter = model.get_best_iteration()
    del model
    gc.collect()

    # Penalite si precision trop basse
    if metrics["precision"] < MIN_PRECISION:
        loss = 1.0
    else:
        loss = -metrics["pr_auc"]  # Hyperopt minimise → negatif

    # Hyperopt log : params plats pour le resultat
    flat_params = {
        "depth": depth,
        "learning_rate": params["learning_rate"],
        "l2_leaf_reg": params["l2_leaf_reg"],
        "random_strength": params["random_strength"],
        "min_data_in_leaf": min_data_in_leaf,
        "border_count": border_count,
        "scale_pos_weight": params["scale_pos_weight"],
        "bootstrap_type": bootstrap_type,
    }
    if bootstrap_type == "Bayesian":
        flat_params["bagging_temperature"] = bt["bagging_temperature"]
    else:
        flat_params["subsample"] = bt["subsample"]

    return {
        "loss": loss,
        "status": STATUS_OK,
        "metrics": metrics,
        "best_iteration": best_iter,
        "flat_params": flat_params,
    }


print(f"Search space defini : {N_TRIALS} trials, GPU (VRAM-safe)")
print(f"  depth: 4-8 | iterations: {MAX_ITERS} | grow_policy: SymmetricTree")
print(f"  beta={BETA} | contrainte precision >= {MIN_PRECISION}")

Search space defini : 30 trials, GPU (VRAM-safe)
  depth: 4-8 | iterations: 4000 | grow_policy: SymmetricTree
  beta=2.0 | contrainte precision >= 0.3


/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Lancement Hyperopt

In [7]:
trials = Trials()

best_raw = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=N_TRIALS,
    trials=trials,
    rstate=np.random.default_rng(RANDOM_SEED),
)

# Trouver le meilleur trial
best_trial_idx = np.argmin([t["result"]["loss"] for t in trials.trials])
best_result = trials.trials[best_trial_idx]["result"]
best_metrics = best_result["metrics"]
best_params = best_result["flat_params"]

print(f"\nMeilleur trial #{best_trial_idx}")
print(f"  PR AUC    : {best_metrics['pr_auc']:.4f}")
print(f"  Recall    : {best_metrics['recall']:.4f}")
print(f"  Precision : {best_metrics['precision']:.4f}")
print(f"  F2        : {best_metrics['f2']:.4f}")
print(f"  Threshold : {best_metrics['threshold']:.3f}")
print(f"  Params    : {best_params}")

  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

Default metric period is 5 because AUC
 is/are not implemented for GPU



  3%|▎         | 1/30 [00:20<09:42, 20.08s/trial, best loss: -0.7114385852309817]

Default metric period is 5 because AUC
 is/are not implemented for GPU



  7%|▋         | 2/30 [01:22<20:55, 44.84s/trial, best loss: -0.7114385852309817]

...

 93%|█████████▎| 28/30 [28:06<02:34, 77.11s/trial, best loss: -0.7154179261375494]

Default metric period is 5 because AUC
 is/are not implemented for GPU



 97%|█████████▋| 29/30 [29:35<01:20, 80.51s/trial, best loss: -0.7154179261375494]

Default metric period is 5 because AUC
 is/are not implemented for GPU



100%|██████████| 30/30 [30:50<00:00, 61.69s/trial, best loss: -0.7154179261375494]

Meilleur trial #27
  PR AUC    : 0.7154
  Recall    : 0.9233
  Precision : 0.4907
  F2        : 0.7849
  Threshold : 0.160
  Params    : {'depth': 6, 'learning_rate': 0.022665704725668104, 'l2_leaf_reg': 45.74910419682211, 'random_strength': 1.7558856459356962, 'min_data_in_leaf': 82, 'border_count': 177, 'scale_pos_weight': 1.0045661847235805, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.1261304410537456}


## Tableau recapitulatif des trials

In [8]:
rows = []
for i, t in enumerate(trials.trials):
    r = t["result"]
    m = r["metrics"]
    rows.append({
        "trial": i,
        "loss": r["loss"],
        "pr_auc": m["pr_auc"],
        "roc_auc": m["roc_auc"],
        "recall": m["recall"],
        "precision": m["precision"],
        "f1": m["f1"],
        "f2": m["f2"],
        "threshold": m["threshold"],
        "best_iteration": r["best_iteration"],
    })

trials_df = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)
display(trials_df.head(15))

,trial,loss,pr_auc,roc_auc,recall,precision,f1,f2,threshold,best_iteration
27,27,-0.715418,0.715418,0.821547,0.923258,0.490688,0.640805,0.784875,0.159932,3997
25,25,-0.714536,0.714536,0.821421,0.916603,0.497941,0.645316,0.784657,0.209483,3996
4,4,-0.714366,0.714366,0.821256,0.917193,0.496942,0.644623,0.784506,0.192201,3906
2,2,-0.714214,0.714214,0.820926,0.913487,0.501016,0.647113,0.784342,0.189567,1191
19,19,-0.713859,0.713859,0.821568,0.909949,0.508377,0.652314,0.785806,0.333582,3076
28,28,-0.713844,0.713844,0.821375,0.925786,0.489685,0.640555,0.785819,0.278184,3918
24,24,-0.713384,0.713384,0.821041,0.922416,0.492356,0.642022,0.785239,0.238430,2511
21,21,-0.713282,0.713282,0.820831,0.917446,0.496060,0.643943,0.784213,0.221747,3962
20,20,-0.713180,0.713180,0.820815,0.904557,0.512554,0.654337,0.784552,0.250452,1827
26,26,-0.713119,0.713119,0.820936,0.925364,0.487486,0.638570,0.784441,0.197549,3991


## Re-entrainement du meilleur modele + logging MLflow complet

In [9]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay,
)
import tempfile, os

# Reconstruire les params CatBoost
retrain_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": MAX_ITERS,
    "random_seed": RANDOM_SEED,
    "verbose": 200,
    "task_type": "GPU",
    "od_type": "Iter",
    "od_wait": EARLY_STOP,
    "grow_policy": "SymmetricTree",
    "depth": int(best_params["depth"]),
    "learning_rate": best_params["learning_rate"],
    "l2_leaf_reg": best_params["l2_leaf_reg"],
    "random_strength": best_params["random_strength"],
    "min_data_in_leaf": int(best_params["min_data_in_leaf"]),
    "border_count": int(best_params["border_count"]),
    "scale_pos_weight": best_params["scale_pos_weight"],
    "bootstrap_type": best_params["bootstrap_type"],
}
if best_params["bootstrap_type"] == "Bayesian":
    retrain_params["bagging_temperature"] = best_params["bagging_temperature"]
else:
    retrain_params["subsample"] = best_params["subsample"]

best_model = CatBoostClassifier(**retrain_params)
best_model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
)

proba_best = best_model.predict_proba(X_valid)[:, 1]
best_thr = best_metrics["threshold"]
metrics_best = compute_metrics(y_valid, proba_best, best_thr)

print(f"\nModele re-entraine")
for k, v in metrics_best.items():
    print(f"  {k:12s}: {v:.4f}")

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7834556	best: 0.7834556 (0)	total: 21ms	remaining: 1m 23s
200:	test: 0.8130684	best: 0.8130684 (200)	total: 4.44s	remaining: 1m 23s
400:	test: 0.8162428	best: 0.8162428 (400)	total: 8.78s	remaining: 1m 18s
600:	test: 0.8178011	best: 0.8178011 (600)	total: 13s	remaining: 1m 13s
800:	test: 0.8187663	best: 0.8187663 (800)	total: 17.3s	remaining: 1m 8s
1000:	test: 0.8192412	best: 0.8192412 (1000)	total: 21.5s	remaining: 1m 4s
1200:	test: 0.8196109	best: 0.8196129 (1195)	total: 25.8s	remaining: 1m
1400:	test: 0.8199674	best: 0.8199674 (1400)	total: 30.1s	remaining: 55.8s
1600:	test: 0.8202624	best: 0.8202624 (1600)	total: 34.4s	remaining: 51.5s
1800:	test: 0.8204959	best: 0.8204962 (1798)	total: 38.7s	remaining: 47.2s
2000:	test: 0.8206765	best: 0.8206775 (1999)	total: 43s	remaining: 42.9s
2200:	test: 0.8208117	best: 0.8208117 (2200)	total: 47.2s	remaining: 38.6s
2400:	test: 0.8208869	best: 0.8208884 (2397)	total: 51.6s	remaining: 34.4s
2600:	test: 0.8210212	best: 0.8210238 (2596

In [10]:
# --- Log dans MLflow ---
import mlflow.catboost

# Metadonnees dataset (etape 6)
dataset_meta = {
    "dataset_size": len(y),
    "train_size": len(y_train),
    "valid_size": len(y_valid),
    "n_features": len(product15_v2),
    "class_0_count": int((y == 0).sum()),
    "class_1_count": int((y == 1).sum()),
    "class_1_ratio": round(int(y.sum()) / len(y), 4),
    "class_imbalance_ratio": round(int((y == 0).sum()) / int(y.sum()), 4),
}

with mlflow.start_run(run_name="hyperopt_best_recall_catboost"):
    mlflow.set_tags({
        "model_family": "catboost",
        "stage": "hyperopt_recall",
        "tuner": "hyperopt",
        "hyperopt_trial": best_trial_idx,
        "optimization_target": "pr_auc",
        "threshold_method": f"f_beta_{BETA}",
    })

    # Hyperparametres
    for k, v in best_params.items():
        mlflow.log_param(k, v)
    mlflow.log_param("task_type", "GPU")
    mlflow.log_param("n_trials", N_TRIALS)
    mlflow.log_param("beta", BETA)
    mlflow.log_param("min_precision_constraint", MIN_PRECISION)
    mlflow.log_param("tuner", "hyperopt")

    # Metadonnees dataset
    mlflow.log_params({f"data_{k}": v for k, v in dataset_meta.items()})

    # Metriques
    mlflow.log_metrics(metrics_best)
    mlflow.log_metric("best_iteration", best_model.get_best_iteration())

    # Artefacts
    with tempfile.TemporaryDirectory() as tmpdir:
        preds_best = (proba_best >= best_thr).astype(int)

        # Matrice de confusion
        fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_valid, preds_best, ax=ax_cm,
            display_labels=["Non grave", "Grave"],
        )
        ax_cm.set_title(f"Confusion Matrix (seuil={best_thr:.3f})")
        cm_path = os.path.join(tmpdir, "confusion_matrix.png")
        fig_cm.savefig(cm_path, dpi=100, bbox_inches="tight")
        plt.close(fig_cm)
        mlflow.log_artifact(cm_path)

        # Courbe ROC
        fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
        RocCurveDisplay.from_predictions(y_valid, proba_best, ax=ax_roc)
        ax_roc.set_title("ROC Curve — Best Hyperopt")
        roc_path = os.path.join(tmpdir, "roc_curve.png")
        fig_roc.savefig(roc_path, dpi=100, bbox_inches="tight")
        plt.close(fig_roc)
        mlflow.log_artifact(roc_path)

        # Courbe Precision-Recall
        fig_pr, ax_pr = plt.subplots(figsize=(6, 5))
        PrecisionRecallDisplay.from_predictions(y_valid, proba_best, ax=ax_pr)
        ax_pr.axhline(y=MIN_PRECISION, color="red", linestyle="--", label=f"min precision={MIN_PRECISION}")
        ax_pr.axvline(x=metrics_best["recall"], color="green", linestyle="--", alpha=0.5, label=f"recall={metrics_best['recall']:.3f}")
        ax_pr.legend()
        ax_pr.set_title("Precision-Recall Curve")
        pr_path = os.path.join(tmpdir, "precision_recall_curve.png")
        fig_pr.savefig(pr_path, dpi=100, bbox_inches="tight")
        plt.close(fig_pr)
        mlflow.log_artifact(pr_path)

        # Feature importance
        fi = best_model.get_feature_importance()
        fi_df = pd.DataFrame({
            "feature": product15_v2,
            "importance": fi,
        }).sort_values("importance", ascending=False)
        fi_path = os.path.join(tmpdir, "feature_importance.csv")
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)

        # Feature names
        fn_path = os.path.join(tmpdir, "feature_names.txt")
        with open(fn_path, "w") as f:
            f.write("\n".join(product15_v2))
        mlflow.log_artifact(fn_path)

        # Hyperopt trials history
        trials_path = os.path.join(tmpdir, "hyperopt_trials.csv")
        trials_df.to_csv(trials_path, index=False)
        mlflow.log_artifact(trials_path)

    # Modele
    mlflow.catboost.log_model(best_model, artifact_path="model")

print("Run MLflow logge : hyperopt_best_recall_catboost")
print(f"Voir dans MLflow UI > experience '{MLFLOW_EXPERIMENT}'")

2026/02/25 14:57:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run hyperopt_best_recall_catboost at: http://127.0.0.1:5000/#/experiments/4/runs/efe13ed7d0054e3ca33e57eb799ec919
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Run MLflow logge : hyperopt_best_recall_catboost
Voir dans MLflow UI > experience 'hyperopt-catboost-recall'


## Export .cbm + meta.json (pour predictor.py)

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import mlflow
import mlflow.catboost

# --- Config autonome (pas besoin d'executer les cellules precedentes) ---
def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable")

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
HYPEROPT_RUN_ID = "efe13ed7d0054e3ca33e57eb799ec919"

FEATURES = [
    "dep", "lum", "atm", "catr", "agg", "int", "circ", "col",
    "vma_bucket", "catv_family_4", "manv_mode", "driver_age_bucket",
    "choc_mode", "driver_trajet_family", "time_bucket",
]

# --- Chargement depuis MLflow ---
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.MlflowClient()
run = client.get_run(HYPEROPT_RUN_ID)

print("Chargement du modele depuis MLflow...")
model = mlflow.catboost.load_model(f"runs:/{HYPEROPT_RUN_ID}/model")

# Recuperer metriques et params du run
metrics = run.data.metrics
params = run.data.params
threshold = metrics.get("threshold", 0.5)

# --- Export .cbm ---
root = _find_root("out")
model_name = "catboost_hyperopt_best"
cbm_path = root / "model" / f"{model_name}.cbm"
meta_path = root / "artifacts" / f"{model_name}_meta.json"
(root / "model").mkdir(exist_ok=True)
(root / "artifacts").mkdir(exist_ok=True)

model.save_model(str(cbm_path))

# --- Export meta.json ---
catboost_param_keys = [
    "depth", "learning_rate", "l2_leaf_reg", "random_strength",
    "min_data_in_leaf", "border_count", "scale_pos_weight",
    "bootstrap_type", "bagging_temperature", "subsample",
]
catboost_params = {}
for k in catboost_param_keys:
    if k in params:
        val = params[k]
        try:
            val = float(val)
            if val == int(val):
                val = int(val)
        except (ValueError, OverflowError):
            pass
        catboost_params[k] = val

meta = {
    "model_name": model_name,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "threshold": threshold,
    "features": FEATURES,
    "cat_features": FEATURES,
    "catboost_params": catboost_params,
    "metrics": {
        k: round(v, 6) for k, v in metrics.items()
        if k in ("threshold", "pr_auc", "roc_auc", "recall",
                 "precision", "f1", "f2", "accuracy")
    },
    "mlflow_run_id": HYPEROPT_RUN_ID,
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"Modele exporte : {cbm_path}")
print(f"Meta exporte   : {meta_path}")
print(f"Threshold      : {threshold:.4f}")
print(f"\nUtilisation dans predictor.py :")
print(f"  MODEL_PATH={cbm_path}")
print(f"  META_PATH={meta_path}")

## Visualisation convergence Hyperopt

In [11]:
import plotly.express as px

# Historique de convergence
losses = [-t["result"]["loss"] if t["result"]["loss"] < 0 else 0 for t in trials.trials]
best_so_far = np.maximum.accumulate(losses)

conv_df = pd.DataFrame({
    "trial": range(len(losses)),
    "PR AUC": losses,
    "Best PR AUC": best_so_far,
})

fig = px.line(
    conv_df, x="trial", y=["PR AUC", "Best PR AUC"],
    title="Hyperopt — Convergence PR AUC",
    labels={"trial": "Trial #", "value": "PR AUC"},
)
fig.show()

# Distribution des hyperparametres importants
param_df = pd.DataFrame([
    t["result"]["flat_params"] | {"pr_auc": t["result"]["metrics"]["pr_auc"]}
    for t in trials.trials
])

for col in ["depth", "learning_rate", "scale_pos_weight", "l2_leaf_reg"]:
    if col in param_df.columns:
        fig = px.scatter(
            param_df, x=col, y="pr_auc",
            title=f"PR AUC vs {col}",
            opacity=0.7,
        )
        fig.show()

## Resume final

In [12]:
print("=" * 60)
print("MEILLEUR MODELE HYPEROPT — FOCUS RECALL")
print("=" * 60)
print(f"  PR AUC     : {metrics_best['pr_auc']:.4f}")
print(f"  ROC AUC    : {metrics_best['roc_auc']:.4f}")
print(f"  Recall     : {metrics_best['recall']:.4f}  <- objectif principal")
print(f"  Precision  : {metrics_best['precision']:.4f}  <- contrainte >= {MIN_PRECISION}")
print(f"  F1         : {metrics_best['f1']:.4f}")
print(f"  F2         : {metrics_best['f2']:.4f}  <- beta={BETA}")
print(f"  Seuil      : {metrics_best['threshold']:.3f}")
print(f"  Best iter  : {best_model.get_best_iteration()}")
print(f"\nHyperparametres :")
for k, v in sorted(best_params.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")
print(f"\n--- Comparaison dans MLflow UI ---")
print(f"Optuna   : experience 'optuna-catboost-recall'")
print(f"Hyperopt : experience 'hyperopt-catboost-recall'")
print(f"URL : {MLFLOW_TRACKING_URI}/#/experiments")

MEILLEUR MODELE HYPEROPT — FOCUS RECALL
  PR AUC     : 0.7152
  ROC AUC    : 0.8215
  Recall     : 0.9223  <- objectif principal
  Precision  : 0.4904  <- contrainte >= 0.3
  F1         : 0.6403
  F2         : 0.7842  <- beta=2.0
  Seuil      : 0.160
  Best iter  : 3999

Hyperparametres :
  bagging_temperature: 0.126130
  bootstrap_type: Bayesian
  border_count: 177
  depth: 6
  l2_leaf_reg: 45.749104
  learning_rate: 0.022666
  min_data_in_leaf: 82
  random_strength: 1.755886
  scale_pos_weight: 1.004566

--- Comparaison dans MLflow UI ---
Optuna   : experience 'optuna-catboost-recall'
Hyperopt : experience 'hyperopt-catboost-recall'
URL : http://127.0.0.1:5000/#/experiments
